### Let's build the rdf graph with the provided csv

In [1]:
import pandas as pd
import urllib.parse
from rdflib import Graph, Literal, RDF, RDFS, URIRef, Namespace
from rdflib.namespace import XSD

df = pd.read_csv("Assignment2.csv")

g = Graph()

SCHEMA = Namespace("https://schema.org/")
EX = Namespace("http://example.org/webshop/")
PRODUCT = Namespace("http://example.org/webshop/product/")
BRAND = Namespace("http://example.org/webshop/brand/")
SUBCAT = Namespace("http://example.org/webshop/subcategory/")
CAT = Namespace("http://example.org/webshop/category/")

g.bind("schema", SCHEMA)
g.bind("ex", EX)
g.bind("product", PRODUCT)
g.bind("brand", BRAND)
g.bind("subcat", SUBCAT)
g.bind("cat", CAT)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)

category_map = {
    'InkCartridge': 'Supplies', 'TonerCartridge': 'Supplies',
    'Plotter': 'Printers', 'InkjetPrinter': 'Printers', 'LaserPrinter': 'Printers',
    'BudgetLaptop': 'Laptops', 'BusinessLaptop': 'Laptops', 'GamingLaptop': 'Laptops',
    'GamingDesktop': 'Desktops', 'Workstation': 'Desktops'
}

for subc_name, cat_name in category_map.items():
    subc_uri = SUBCAT[urllib.parse.quote(subc_name.replace(" ", "_"))]
    cat_uri = CAT[urllib.parse.quote(cat_name)]
    g.add((subc_uri, RDFS.subClassOf, cat_uri))

for _, row in df.iterrows():
    raw_sku = str(row['schema_sku_value']).strip()
    clean_sku = urllib.parse.quote(raw_sku.replace(" ", "_"))

    item_uri = PRODUCT[clean_sku]
    brand_name = str(row['brand'])
    brand_uri = BRAND[urllib.parse.quote(brand_name)]
    subcat_name = str(row['subcategory']).replace(" ", "_")
    subcat_uri = SUBCAT[urllib.parse.quote(subcat_name)]

    g.add((item_uri, RDF.type, subcat_uri))
    g.add((item_uri, SCHEMA.sku, Literal(raw_sku, datatype=XSD.string)))
    g.add((item_uri, SCHEMA.name, Literal(str(row['item_name']), datatype=XSD.string)))
    g.add((item_uri, SCHEMA.url, URIRef(str(row['schema_url_value']))))
    g.add((item_uri, SCHEMA.price, Literal(float(row['schema_price_value']), datatype=XSD.decimal)))

    if pd.notnull(row['schema_color_value']):
        g.add((item_uri, SCHEMA.color, Literal(str(row['schema_color_value']), datatype=XSD.string)))

    g.add((item_uri, SCHEMA.brand, brand_uri))
    g.add((brand_uri, RDF.type, SCHEMA.Brand))
    g.add((brand_uri, SCHEMA.name, Literal(brand_name, datatype=XSD.string)))

g.serialize(destination="Group-6_ken3140_webshop.ttl", format="turtle")
print(f"RDF Graph created with {len(g)} triples and saved as 'Group-6_ken3140_webshop.ttl'.\n")


RDF Graph created with 370 triples and saved as 'Group-6_ken3140_webshop.ttl'.



### Adding the ontology to the populated rdfs graph for more freedom in queries
eg. select product instead of having to union everything.

In [2]:

from rdflib import Graph, Literal, RDF, RDFS, OWL, URIRef, Namespace

g = Graph()
g.parse("Group-6_ken3140_webshop.ttl", format="turtle")
g.parse("ontology.ttl", format="turtle")

SUBCAT = Namespace("http://example.org/webshop/subcategory/")
HP = Namespace("https://www.hp.com/ontology#")
SCHEMA = Namespace("https://schema.org/")

mappings = [
    (SUBCAT.GamingLaptop, RDFS.subClassOf, HP.GamingLaptop),
    (SUBCAT.BusinessLaptop, RDFS.subClassOf, HP.BusinessLaptop),
    (SUBCAT.BudgetLaptop, RDFS.subClassOf, HP.BudgetLaptop),
    (SUBCAT.GamingDesktop, RDFS.subClassOf, HP.GamingDesktop),
    (SUBCAT.Workstation, RDFS.subClassOf, HP.Workstation),
    (SUBCAT.InkjetPrinter, RDFS.subClassOf, HP.InkjetPrinter),
    (SUBCAT.LaserPrinter, RDFS.subClassOf, HP.LaserPrinter),
    (SUBCAT.Plotter, RDFS.subClassOf, HP.Plotter),
    (SUBCAT.InkCartridge, RDFS.subClassOf, HP.Ink),
    (SUBCAT.TonerCartridge, RDFS.subClassOf, HP.Ink),
    (SCHEMA.price, OWL.equivalentProperty, HP.price),
    (SCHEMA.brand, OWL.equivalentProperty, HP.hasBrand)
]

for triple in mappings:
    g.add(triple)
g.serialize(destination="Group-6_ken3140_webshop.ttl", format="turtle")

print(f"Merged Graph contains {len(g)} total triples.\n")

Merged Graph contains 462 total triples.



# Queries
a) For a given item (select a random item from your RDF Graph), provide all its
categories and subcategories, and its brand.


In [3]:
from rdflib import Graph, Literal
from rdflib.namespace import XSD

def run_query(query_str):
    res = g.query(query_str)
    for row in res:
        print([str(val) for val in row])
    print("\n")




# sku = input("Please enter a product SKU: ").strip()
sku = "B7RT9AE"

query = """
PREFIX schema: <https://schema.org/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?product ?productName ?brandName ?category WHERE {
    ?product schema:sku ?targetSku ;
             schema:name ?productName ;
             schema:brand ?brandResource .

    ?brandResource schema:name ?brandName .

    ?product a/rdfs:subClassOf* ?category .
}
"""# efficient because we first find the only object with a unique sku value then proceed to take information from it

results = g.query(
    query,
    initBindings={'targetSku': Literal(sku, datatype=XSD.string)}
)
for row in results:
    print([str(val) for val in row])
print("\n")

['http://example.org/webshop/product/B7RT9AE', 'HP 302XL/304XL originele zwarte inktcartridge', 'HP', 'http://example.org/webshop/subcategory/InkCartridge']
['http://example.org/webshop/product/B7RT9AE', 'HP 302XL/304XL originele zwarte inktcartridge', 'HP', 'http://example.org/webshop/category/Supplies']
['http://example.org/webshop/product/B7RT9AE', 'HP 302XL/304XL originele zwarte inktcartridge', 'HP', 'https://www.hp.com/ontology#Ink']
['http://example.org/webshop/product/B7RT9AE', 'HP 302XL/304XL originele zwarte inktcartridge', 'HP', 'https://schema.org/Product']




b) Provide items from different subcategories that have the same brand.


In [ ]:
query_b = """
PREFIX schema: <https://schema.org/>

SELECT ?brandName ?item1 ?subcat1 ?item2 ?subcat2 WHERE {
    ?product1 schema:brand ?brand ;
              a ?subcat1 ;
              schema:name ?item1 .
    ?product2 schema:brand ?brand ;
              a ?subcat2 ;
              schema:name ?item2 .
    ?brand schema:name ?brandName .
    FILTER (STR(?subcat1) < STR(?subcat2))
}
ORDER BY ?brandName
"""# we used < instead of != so every pair only shows up once. It was hard to spot the mistake at the beginning.

run_query(query_b)

['HP', 'HP 302XL/304XL originele drie-kleuren inktcartridge', 'http://example.org/webshop/subcategory/InkCartridge', 'HP Smart Tank 6005 All-in-One', 'http://example.org/webshop/subcategory/InkjetPrinter']
['HP', 'HP 302XL/304XL originele zwarte inktcartridge', 'http://example.org/webshop/subcategory/InkCartridge', 'HP Smart Tank 6005 All-in-One', 'http://example.org/webshop/subcategory/InkjetPrinter']
['HP', 'HP 301 originele drie-kleuren inktcartridge', 'http://example.org/webshop/subcategory/InkCartridge', 'HP Smart Tank 6005 All-in-One', 'http://example.org/webshop/subcategory/InkjetPrinter']
['HP', 'HP 301XL originele high-capacity zwarte inktcartridge', 'http://example.org/webshop/subcategory/InkCartridge', 'HP Smart Tank 6005 All-in-One', 'http://example.org/webshop/subcategory/InkjetPrinter']
['HP', 'HP 303 originele drie-kleuren inktcartridge', 'http://example.org/webshop/subcategory/InkCartridge', 'HP Smart Tank 6005 All-in-One', 'http://example.org/webshop/subcategory/Inkjet

c) Group products by brand and show the average price or rating for each brand.


In [5]:
query_c = """
PREFIX schema: <https://schema.org/>

SELECT ?brandName (COUNT(?product) AS ?products) (ROUND(AVG(?price)) AS ?avgPrice) WHERE {
    ?product schema:brand ?brand ;
             schema:price ?price .
    ?brand schema:name ?brandName .
}
GROUP BY ?brandName
ORDER BY DESC(?avgPrice)
"""

run_query(query_c)

['HP_Z', '6', '4599']
['HyperX', '7', '3256']
['OMEN', '2', '2099']
['DesignJet', '5', '1950']
['ProBook', '4', '1374']
['Victus', '1', '1199']
['OmniBook', '5', '829']
['LaserJet', '9', '668']
['OfficeJet', '1', '249']
['Envy', '2', '112']
['HP', '7', '71']
['DeskJet', '1', '60']




d) Sort products in one category according to average brand price or rating.


In [6]:
query_d = """
PREFIX schema: <https://schema.org/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX hp: <https://www.hp.com/ontology#>

SELECT ?productName ?brandName ?price ?brandAvgPrice WHERE {
    {
        SELECT ?brand (ROUND(AVG(?p)) AS ?brandAvgPrice) WHERE {
            ?x a/rdfs:subClassOf* hp:Laptop ;
               schema:brand ?brand ;
               schema:price ?p .
        }
        GROUP BY ?brand
    }
    ?product a/rdfs:subClassOf* hp:Laptop ;
             schema:brand ?brand ;
             schema:name ?productName ;
             schema:price ?price .
    ?brand schema:name ?brandName .
}
ORDER BY DESC(?brandAvgPrice) DESC(?price)
"""# subquery gets the average price per brand within laptops, then every laptop is sorted by it

run_query(query_d)

['HP Zbook Ultra G1a 14" mobile workstation - OLED scherm met touch - AMD Radeon 8050S Graphics', 'HP_Z', '3299.0', '3299']
['HP OMEN Transcend 14" Gaming Laptop - 14-fb1002nb - Shadow Black - QHD OLED - RTX 5060 - Azerty toetsenbord met RGB verlichting per toets', 'OMEN', '2999.0', '2999']
['HP HyperX OMEN MAX 16 inch Gaming Laptop PC 16-ak1002nb', 'HyperX', '2799.0', '1966']
['HP HyperX OMEN 15 inch Gaming Laptop PC 15-gb0000nb', 'HyperX', '1699.0', '1966']
['HP HyperX OMEN 16 inch Gaming Laptop PC 16-ap1004nb', 'HyperX', '1399.0', '1966']
['HP ProBook 4 G2i 16" Next Gen AI laptop met touch en LTE/5G - Azerty toetsenbord met verlichting - 3 jaar onsite hardware support', 'ProBook', '1649.0', '1374']
['HP ProBook 4 G1i 16 inch Notebook AI PC Wolf Pro Security Edition', 'ProBook', '1449.0', '1374']
['HP ProBook 4 G1i 16" laptop – LTE/4G - Azerty toetsenbord met verlichting - 3 jaar onsite hardware support', 'ProBook', '1299.0', '1374']
['HP ProBook 4 G1q 14" laptop', 'ProBook', '1098.9

e) Using an external service point (e.g. https://query.wikidata.org/), provide a
description of 5 facts about the top ranked brand from part D, e.g. location of
headquarters. You may return images as one of your facts.


In [ ]:
top_brand = str(list(g.query(query_d))[0].brandName).replace("_", " ")
print("top brand from d:", top_brand)

query_e = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX sdo: <http://schema.org/>

SELECT ?description
       (GROUP_CONCAT(DISTINCT ?typeLabel; separator=", ") AS ?type)
       (GROUP_CONCAT(DISTINCT ?manufacturerLabel; separator=", ") AS ?manufacturer)
       (GROUP_CONCAT(DISTINCT ?hqLabel; separator=", ") AS ?headquarters)
       ?image
WHERE {
    SERVICE <https://query.wikidata.org/sparql> {
        ?brand rdfs:label ?brandLabel ;
               sdo:description ?description ;
               wdt:P31 ?t ;
               wdt:P176 ?m ;
               wdt:P18 ?image .
        ?t rdfs:label ?typeLabel .
        ?m rdfs:label ?manufacturerLabel ;
           wdt:P159 ?hq .
        ?hq rdfs:label ?hqLabel .
        FILTER (lang(?description) = "en" && lang(?typeLabel) = "en" && lang(?manufacturerLabel) = "en" && lang(?hqLabel) = "en")
    }
}
GROUP BY ?description ?image
"""

results = g.query(query_e, initBindings={'brandLabel': Literal(top_brand, lang="en")})
for row in results:
    print([str(val) for val in row])
print("\n")

top brand from d: HP Z
['series of workstation computer models', 'model series, computer model series', 'Hewlett-Packard, HP Inc.', 'Palo Alto', 'http://commons.wikimedia.org/wiki/Special:FilePath/Six%20HP%20workstations.jpg']




f) Recommend an item which is similar to the item using your linked RDF graph
(i.e., shared properties and categories).


In [8]:
query_f = """
PREFIX schema: <https://schema.org/>

SELECT ?name ?brandName ?color ?price WHERE {
    ?item schema:sku ?targetSku ;
          a ?subcat ;
          schema:brand ?brand ;
          schema:price ?itemPrice .
    ?other a ?subcat ;
           schema:name ?name ;
           schema:brand ?otherBrand ;
           schema:price ?price .
    ?otherBrand schema:name ?brandName .
    OPTIONAL { ?other schema:color ?color }
    OPTIONAL { ?item schema:color ?itemColor }
    FILTER (?other != ?item)
    BIND (IF(?otherBrand = ?brand, 1, 0) AS ?sameBrand)
    BIND (IF(BOUND(?color) && BOUND(?itemColor) && ?color = ?itemColor, 1, 0) AS ?sameColor)
    BIND (ABS(?price - ?itemPrice) AS ?priceDiff)
}
ORDER BY DESC(?sameBrand) DESC(?sameColor) ?priceDiff
LIMIT 3
"""# same subcategory as the item from a), then same brand, same color and closest price first

results = g.query(query_f, initBindings={'targetSku': Literal(sku, datatype=XSD.string)})
for row in results:
    print([str(val) for val in row])
print("\n")

['HP 301XL originele high-capacity zwarte inktcartridge', 'HP', 'Black', '57.49']
['HP 302XL/304XL originele drie-kleuren inktcartridge', 'HP', 'Cyan/Magenta/Yellow', '52.49']
['HP 301 originele drie-kleuren inktcartridge', 'HP', 'Cyan/Magenta/Yellow', '33.49']




g) Write your own question about the webshop in plain English, then translate it to
the corresponding SPARQL query, and run it on the graph. Provide a rationale for
why this query would be valuable in a webshop setting, such as for semantic
search or other applications.

In [9]:
query_g = """
PREFIX schema: <https://schema.org/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX hp: <https://www.hp.com/ontology#>

SELECT ?name ?type ?price WHERE {
    ?printer a ?type ;
             schema:name ?name ;
             schema:price ?price .
    ?type rdfs:subClassOf* hp:Printer .
    FILTER (?price < 300)
}
ORDER BY ?price
"""

run_query(query_g)

['HP DeskJet 4320 All-in-One printer met 6 maanden Instant Ink', 'http://example.org/webshop/subcategory/InkjetPrinter', '60.0']
['HP LaserJet M209d printer', 'http://example.org/webshop/subcategory/LaserPrinter', '99.0']
['HP Envy Photo 7930 All-in-One printer met 6 maanden Instant Ink', 'http://example.org/webshop/subcategory/InkjetPrinter', '110.0']
['HP Envy 6532e All-in-One printer met 9 maanden Instant Ink via HP+', 'http://example.org/webshop/subcategory/InkjetPrinter', '113.99']
['HP Smart Tank 6005 All-in-One', 'http://example.org/webshop/subcategory/InkjetPrinter', '217.0']
['HP LaserJet MFP M235sdw printer', 'http://example.org/webshop/subcategory/LaserPrinter', '239.89']
['HP OfficeJet Pro 9120b All-in-One printer', 'http://example.org/webshop/subcategory/InkjetPrinter', '249.01']




**Question:** Which printers cost less than 300 euro, and what type of printer is each one?

**Why it is useful:** this is what a price filter on a "Printers" page does. Because the subcategories are linked to `hp:Printer` in the ontology, one query searches inkjet printers, laser printers and plotters without listing them. If a new printer subcategory is added later and linked to `hp:Printer`, it is included automatically, which a simple `category = '...'` filter would not do.